# Module 03 — Lecture 3: Spike Train Analysis & Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_03_lif_neurons/03_spike_analysis.ipynb)

---

A simulation is only useful if we can analyse and visualise its output. This lecture covers the standard spike train analysis toolkit used in computational and experimental neuroscience.

**Learning objectives:**
- Create raster plots from GPU spike output
- Compute population firing rates and PSTH
- Analyse inter-spike interval (ISI) distributions
- Compute pairwise spike-train correlations
- Export data for further analysis

In [ ]:
!nvidia-smi

In [ ]:
# First, run the LIF simulation to generate spikes.txt
# (compile lif_gpu from Lecture 2 if not already done)
import os
if not os.path.exists('lif_gpu'):
    print("Compiling lif_gpu...")
    !nvcc -O2 -o lif_gpu lif_gpu.cu -lm

# Run: 10,000 neurons, 2000 ms
!./lif_gpu 10000 2000
print("Simulation complete. spikes.txt written.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Load spike data
spikes = pd.read_csv('spikes.txt', sep=' ', names=['neuron', 'time_ms'])
print(f"Total spikes: {len(spikes):,}")
print(f"Neurons: {spikes['neuron'].nunique():,}")
print(f"Time range: {spikes['time_ms'].min():.1f} – {spikes['time_ms'].max():.1f} ms")
print(f"Mean firing rate: {len(spikes) / spikes['neuron'].nunique() / 2.0:.1f} Hz")

In [ ]:
# ── 1. Raster Plot ────────────────────────────────────────────────────────────
N = 10000
T_ms = 2000

# Show 200 randomly selected neurons for clarity
rng = np.random.default_rng(42)
sample_neurons = rng.choice(N, size=200, replace=False)
sample_spikes = spikes[spikes['neuron'].isin(sample_neurons)]

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

# Top: raster plot
ax_raster = fig.add_subplot(gs[0])
ax_raster.scatter(sample_spikes['time_ms'], sample_spikes['neuron'],
                  s=0.3, c='k', alpha=0.6, linewidths=0)
ax_raster.set_xlim(0, T_ms)
ax_raster.set_ylabel('Neuron ID', fontsize=12)
ax_raster.set_title('LIF Network — Raster Plot (200 of 10,000 neurons)', fontsize=13)
ax_raster.set_xticklabels([])

# Bottom: population firing rate (PSTH)
ax_psth = fig.add_subplot(gs[1])
bin_size_ms = 10
bins = np.arange(0, T_ms + bin_size_ms, bin_size_ms)
counts, _ = np.histogram(spikes['time_ms'], bins=bins)
rate_hz = counts / N / (bin_size_ms / 1000)  # convert to Hz

ax_psth.bar(bins[:-1], rate_hz, width=bin_size_ms * 0.9,
            color='steelblue', alpha=0.8, align='edge')
ax_psth.set_xlim(0, T_ms)
ax_psth.set_xlabel('Time (ms)', fontsize=12)
ax_psth.set_ylabel('Pop. rate (Hz)', fontsize=12)

plt.savefig('raster_psth.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Population firing rate: {rate_hz.mean():.1f} ± {rate_hz.std():.1f} Hz")

In [ ]:
# ── 2. ISI Distribution ───────────────────────────────────────────────────────
# Compute ISIs for a sample of neurons

all_isis = []
cvs = []
sample_ids = rng.choice(N, size=500, replace=False)

for nid in sample_ids:
    neuron_spikes = spikes[spikes['neuron'] == nid]['time_ms'].values
    if len(neuron_spikes) >= 3:
        isis = np.diff(np.sort(neuron_spikes))
        all_isis.extend(isis)
        cvs.append(np.std(isis) / np.mean(isis))

all_isis = np.array(all_isis)
cvs = np.array(cvs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(all_isis, bins=100, range=(0, 200), color='steelblue', alpha=0.8, edgecolor='none')
ax1.set_xlabel('ISI (ms)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Inter-Spike Interval Distribution\n(500 sampled neurons)', fontsize=12)
ax1.axvline(np.median(all_isis), color='r', linestyle='--',
            label=f'Median ISI = {np.median(all_isis):.1f} ms')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.hist(cvs, bins=40, color='tomato', alpha=0.8, edgecolor='none')
ax2.set_xlabel('Coefficient of Variation (CV)', fontsize=12)
ax2.set_ylabel('Number of neurons', fontsize=12)
ax2.set_title('ISI CV Distribution\n(heterogeneous inputs → spread of CVs)', fontsize=12)
ax2.axvline(1.0, color='k', linestyle='--', alpha=0.5, label='CV=1 (Poisson)')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('isi_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Median ISI: {np.median(all_isis):.1f} ms  |  Mean CV: {cvs.mean():.3f}")

In [ ]:
# ── 3. Firing Rate Distribution ───────────────────────────────────────────────
# Per-neuron firing rate histogram
neuron_counts = spikes.groupby('neuron').size()
all_neurons = pd.Series(0, index=range(N))
all_neurons.update(neuron_counts)
firing_rates_hz = all_neurons / (T_ms / 1000.0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(firing_rates_hz, bins=60, color='mediumpurple', alpha=0.85, edgecolor='none')
ax.set_xlabel('Firing Rate (Hz)', fontsize=13)
ax.set_ylabel('Number of neurons', fontsize=13)
ax.set_title(f'Per-Neuron Firing Rate Distribution\nN={N}, T={T_ms} ms, I heterogeneous', fontsize=13)
ax.axvline(firing_rates_hz.mean(), color='r', linestyle='--',
           label=f'Mean = {firing_rates_hz.mean():.1f} Hz')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('firing_rate_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Firing rates: mean={firing_rates_hz.mean():.1f} Hz, "
      f"std={firing_rates_hz.std():.1f} Hz, "
      f"range=[{firing_rates_hz.min():.0f}, {firing_rates_hz.max():.0f}] Hz")

## Summary

Standard spike train analysis pipeline:

| Analysis | What it shows | Key metric |
|----------|---------------|------------|
| Raster plot | When each neuron fires | Visual pattern detection |
| PSTH | Population activity over time | Mean population rate (Hz) |
| ISI distribution | Regularity of individual neurons | CV (0=regular, 1=Poisson-like) |
| Firing rate distribution | Heterogeneity across the network | Mean ± std (Hz) |

**Proceed to:** [Exercise 03](exercises/ex03_stub.ipynb) — add a refractory period and heterogeneous currents with noise.

---

**Next →** [Module 04 — Hodgkin-Huxley Theory](../module_04_hodgkin_huxley/01_hh_theory.ipynb) &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_04_hodgkin_huxley/01_hh_theory.ipynb)